# <u>Evaluation</u>

### Prerequisites:
* <a href="Supervised ML Basis.ipynb">Check out the notebook on Supervised ML Basis</a>


## Topics

* [1. Goal](#goal)
* [2. Generalization error of a Learner and fixed model](#GE)
* [3. Inner vs Outer loss](#in_out)
* [4. Training error vs. Test error](#train_test)
* [5. Over-and Underfitting](#over_under)
* [6. Metrics](#metric)
    * [6.1 Regression](#regression)
    * [6.2 Classification](#classification)
* [7. Resampling](#resapmling)
    * [7.1 Hold-out sampling](#hold_out)
    * [7.2 k-fold Cross Validation (CV)](#CV)
    * [7.3 Cross-Validation-Stratification](#CV_strat)
    * [7.4 Leave-one-out CV (LOO-CV)](#LOO-CV)
    * [7.5 Leave-one-object-out](#LOOO)
    * [7.6 Bootstrap](#bootstrap)
* [8. Choosing the right metric](#metric)


In [11]:
import numpy as np # for arrays and random numbers
from sklearn.metrics import mean_squared_error as mse # mean squarred error metric
from sklearn.datasets import make_regression # create toy data for Regression
from sklearn.datasets import make_classification # create toy data for Classification
from sklearn.datasets import make_blobs # create toy data also for Classification
import matplotlib.pyplot as plt # for plotting
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting
from sklearn.model_selection import train_test_split # split dataset into train and test set
from sklearn.model_selection import StratifiedKFold # CV Stratification
from sklearn.metrics import (
    mean_squared_error, # MSE
    mean_absolute_error, # MAE
    mean_absolute_percentage_error, # MAPE
    r2_score # R^2
)
from sklearn.metrics import (
    accuracy_score, # Accuracy
    brier_score_loss, # Brier Score
    log_loss, # Log Loss
    confusion_matrix, # TP, FP, FN, TN
    precision_score, # Precision = PPV
    recall_score, # Recall = TPR
    roc_auc_score
)
print("Setup complete")

Setup complete


<a class="anchor" id="goal"></a>
# 1. Goal

- Estimate Generalization Error ($\text{GE}$)

- $\text{GE}(\hat{f},L):=\mathbb{E}\left[L\left(y,\hat{f}(x)\right)\right] \overset{(x,y)\sim \mathbb{P}_{xy}}{\Longrightarrow}$ Expected future model loss ($\mathbb{P}_{xy}$ is data generating process)

- Estimate $\text{GE}(\hat{f},L)$ with $\widehat{\text{GE}}(\hat{f},L):=\frac{1}{m} \sum_{(x,y) \in \mathcal{D}} \left[L\left(y,\hat{f}(x)\right)\right]$ where $| \mathcal{D}| = m$

- Use $(x,y) \in \mathcal{D}_\text{test}$ since if $(x,y) \in \mathcal{D}_\text{train}$ the estimate $\widehat{\text{GE}}(\hat{f},L)$ will be biased

- Problem: No access to unseen data $\Rightarrow$ Divide dataset $\mathcal{D}$ into $\mathcal{D}_\text{train}$ and $\mathcal{D}_\text{test}$

- $\widehat{\text{GE}}(\hat{f},L):= \frac{1}{m} \sum_{(x,y) \in \mathcal{D}_\text{test}} \left[L\left(y,\hat{f}(x)\right)\right]$ where $| \mathcal{D}_\text{test}| = m$ will be unbiased

- $L\left(y,\hat{f}(x)\right)$ always indicates how well the target $y$ mathces the prediction $\hat{f}(x)$

```python
from sklearn.model_selection import train_test_split # to split dataset into train and test set

X_train, X_test, y_train, y_test = train_test_split(
    X, y, # features and target
    test_size=0.2, # proportion used for testing (20%)
    train_size=0.8, # proportion used for training (80%) (optional if test_size given)
    random_state=1414, # ensures reproducible split
    shuffle=True, # randomly shuffle data before splitting
    stratify=y # preserve class proportions in y (useful for classification)
)
```

$$
\overset{\mathcal{D}}{
\begin{array}{ccccccc}
& x_1 & x_2 & \ldots & x_p & y & \hat{y} \\
\hline

& \color{green}x_{11} & \color{green}x_{12} & \ldots & \color{green}x_{1p} & \color{green}y_1 & \\

& \color{green}x_{21} & \color{green}x_{22} & \ldots & \color{green}x_{2p} & \color{green}y_2 & \\

\color{green}\mathcal{D}_\text{train}
& \color{green}x_{31} & \color{green}x_{32} & \ldots & \color{green}x_{3p} & \color{green}y_3 & \\

& \vdots & \vdots & \ddots & \vdots & \vdots & \\

& \color{green}x_{(n-3)1} & \color{green}x_{(n-3)2} & \ldots & \color{green}x_{(n-3)p} & \color{green}y_{n-3} & \\

\hline

& \color{red}x_{(n-2)1} & \color{red}x_{(n-2)2} & \ldots & \color{red}x_{(n-2)p} & \color{red}y_{n-2} & \hat{y} \\

\color{red}\mathcal{D}_\text{test}
& \color{red}x_{(n-1)1} & \color{red}x_{(n-1)2} & \ldots & \color{red}x_{(n-1)p} & \color{red}y_{n-1} & \hat{y} \\

& \color{red}x_{n1} & \color{red}x_{n2} & \ldots & \color{red}x_{np} & \color{red}y_n & \hat{y} \\
\end{array}
}
$$

<p align="center">
<img src="pics/1.png" width="600"/>
</p>



<a class="anchor" id="GE"></a>
# 2. Generalization error of a Learner and fixed model

<div style="
<-- -->background-color:rgb(239, 201, 50);;
padding:16px;
border-radius:8px;
color:white;
">

### GE of a Learner $\mathcal{I}$

- Distribution of different models $\hat{f}$ trained on different training datasets

<div style="
<-- -->background-color:rgb(213, 230, 222);
padding:14px;
border-radius:8px;
color:white;
margin-top:12px;
margin-left:24px;
width:85%;
border-left:4px solid rgb(208, 215, 212);
box-shadow: inset 0 0 8px rgba(219, 207, 207, 0.3);
">

### GE of a fixed model

- Single random draw of a fitted model $\hat{f}$
- $\widehat{\text{GE}}(\hat{f},L):= \frac{1}{m} \sum_{(x,y) \in \mathcal{D}_\text{test}}\left[L\left(y,\hat{f}(x)\right)\right] \Rightarrow$ Expected error on unseen data
- "How good is this model outside the training data?"


</div>

- $\text{GE}=$ Expected error of all models $\hat{f}$ over all training datasets
- "How good is the model in the long run?"

</div>

<a class="anchor" id="in_out"></a>
# 3. Inner vs Outer loss

<div style="
<-- -->background-color:#2f4154;
padding:16px;
border-radius:4px;
color:white;
">

### Outer loss (evaluation loss)

- Empirical estimator of the outer/generalization loss on $\mathcal{D}_\text{test}$
- Measures error on unseen data $\mathcal{D}_\text{test}$
- Examples:
    - Mean squarred error: $\rho_\text{MSE}(y,\mathcal{F})=\frac{1}{m} \sum_{i=1}^m (y^{(i)}- \hat{y}^{(i)})^2$ with $\hat{y}^{(i)}=\hat{\theta}^\top x^{(i)}$
    - Mean absolute error: $\rho_\text{MAE}(y,\mathcal{F})=\frac{1}{m} \sum_{i=1}^m \mid y^{(i)}- \hat{y}^{(i)} \mid$
    - Missclassification error: $\rho_\text{MCE}(y,\mathcal{F})=\frac{1}{m} \sum_{i=1}^m [y^{(i)} \neq \hat{y}^{(i)}]$

<div style="
<-- -->background-color:#243341;
padding:14px;
border-radius:8px;
color:white;
margin-top:12px;
margin-left:24px;
width:85%;
border-left:4px solid #7aa6d1;
box-shadow: inset 0 0 8px rgba(0,0,0,0.3);
">

### Inner loss 

- Function the learner actually minimizes during training
- Function chosen because it is differentiable/convex
- Examples:
    - Ordinary least squares: $\hat{\theta}=\arg \min_\theta \sum_{i=1}^m (y^{(i)} -\theta^\top x^{(i)})^2$
    - Log-loss: $L(y,\pi(x \mid \theta))=-y\log(\pi(x \mid \theta)) - (1-y)\log((1-\pi(x \mid \theta)))$
    - Hinge loss: $\max(0,1-y^{(i)}(w^\top x^{(i)} +b))$

**Dangers of using inner loss as outer loss:**
- Surrogate losses live on an arbitrary numerical scale unrelated to the practical target criterion.
    - i.e. hinge loss = 0.4 does not translate to 40% badness
- some inner losses are model-specific surrogate objectives, making cross-model comparison numerically meaningless
    - i.e. log-loss only applies to probabilistic classifier producing estimated class probabilities
</div>

- Minimizing inner loss is intended to indirectly reduce outer loss

</div>

<a class="anchor" id="train_test"></a>
# 4. Training error vs. Test error


<div style="display:flex; gap:5px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--  -->border:2px solid white;
width:50%;
">

<h5 style="text-align:center;">Training error</h5>

$$
\rho(y_\text{train}, \mathcal{F}_\text{train}), 
\mathcal{F}_\text{train}=
\begin{bmatrix} 
\hat{f}_{\mathcal{D}_\text{train}}(x_\text{train}^{(i)}) \\
\vdots \\
\hat{f}_{\mathcal{D}_\text{train}}(x_\text{train}^{(m)}) \\
\end{bmatrix}
$$

- Train error: Often biased (overfitting risk)
- Interpolator = Model that passes exactly through all training points
- Example: Polynomial Regression
    - Polynomial degree $d$ increases $\Rightarrow$ $\rho_\text{MSE}$ on $\mathcal{D}_\text{train}$ decreases 
- Training error decreases with $\begin{cases} \text{smaller training sets} \\ \text{increasing model complexity}  \end{cases}$

<p align="center">
<img src="pics/2.png" width="600"/>
</p>

</div>

<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--  -->border:2px solid white;
width:50%;
">

<h5 style="text-align:center;">Test error</h5>

$$
\rho(y_\text{test}, \mathcal{F}_\text{test}), 
\mathcal{F}_\text{test}=
\begin{bmatrix} 
\hat{f}_{\mathcal{D}_\text{test}}(x_\text{test}^{(i)}) \\
\vdots \\
\hat{f}_{\mathcal{D}_\text{test}}(x_\text{test}^{(m)}) \\
\end{bmatrix}
$$

- Test error: Approximately unbiased when evaluated on many unseen observations drawn from the same population
- Example: Polynomial Regression
    - Polynomial degree $d$ increases $\Rightarrow$ $\rho_\text{MSE}$ on $\mathcal{D}_\text{test}$ forms a $\mathrm{U}$-shape

- Test error decreases with $\rightarrow$ Larger training sets
- Test error $\rightarrow$ Higher variance $\begin{cases} \text{with smaller test sets} \\ \text{with increasing model complexity}  \end{cases}$


<p align="center">
<img src="pics/3.png" width="600"/>
</p>

</div>
</div>

$\underbrace{\text{Hold-out-sampling: Trade-off} \begin{cases} \text{Bigger training set} \rightarrow \text{Lower Bias (better model), higher variance (small test set)} 
\\ 
\text{Bigger test set} \rightarrow \text{Higher Bias (weak model), lower variance (stable test error)}  \end{cases}}_{\text{split ratio determines which effect dominates}}$

- $\text{Bias}(x)=\mathbb{E}_\text{train}\left[\hat{f}(x)\right] - f(x)$
    - $\mathbb{E}_\text{train}\left[\hat{f}(x)\right]$: average prediction of model trained trained across all possible training sets
    - $f(x)$: true function value
- Variance =  Measure of how much the predictions of the model change when trained on different training sets

<a class="anchor" id="over_under"></a>
# 5. Over-and Underfitting

<div style="display:flex; gap:5px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--  -->border:2px solid white;
width:50%;
">

<h5 style="text-align:center;">Overfitting</h5>

- Small train errors & high test error
- $\text{OF}(\hat{f},L):=\underbrace{\text{GE}(\hat{f},L)}_{\text{(theoretical) GE}}-\underbrace{\mathcal{R}(\hat{f},L)}_{\text{training errors}}$
- Low bias & high variance
- $\text{OF}(\hat{f},L)$ influenced by
    - Complexity of hypothesis space
    - Training set size
    - Dimensionality of feature space


</div>

<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--  -->border:2px solid white;
width:50%;
">

<h5 style="text-align:center;">Underfitting</h5>

- High train errors & High test error
- $\text{UF}(\hat{f},L):=\text{GE}(\hat{f},L)-\underbrace{\text{GE}(f^*,L)}_{\text{Bayes optimal model}}$
- High vias & low variance

</div>
</div>


<a class="anchor" id="metric"></a>
# 6. Metrics

<a class="anchor" id="regression"></a>
## 6.1 Regression

* SSE: $\rho_\text{SSE}(y,\mathcal{F})=\sum_{i=1}^m (y^{(i)}-\hat{y}^{(i)})^2$

* MSE: $\rho_\text{MSE}(y,\mathcal{F})=\frac{1}{m}\rho_\text{SSE}(y,\mathcal{F})$ (L2-loss)

* Root MSE: $\rho_\text{RMSE}(y,\mathcal{F})=\sqrt{\rho_\text{MSE}(y,\mathcal{F})}$


* MAE: $\rho_\text{MAE}(y,\mathcal{F})=\frac{1}{m}\sum_{i=1}^m \mid y^{(i)}-\hat{y}^{(i)} \mid [0,\infty) $ (L1-loss)

* MAPE: $\rho_\text{MAPE}(y,\mathcal{F})=\frac{1}{m} \mid \frac{y^{(i)}-\hat{y}^{(i)}}{y^{(i)}} \mid$ 

* $R^2$: $\rho_{R^2}(y,\mathcal{F})=1-\frac{\text{SSE}}{\text{SST}}$ where $\text{SST}=\sum_{i=1}^m (y^{(i)}-\bar{y})^2$

* $\rho_\text{Spearmann}(y,\mathcal{F})=\frac{\mathbb{Cov}(\text{rg}(y),\text{rg}(\hat{y}))}{\sqrt{\mathbb{Var}(\text{rg}(y))}\sqrt{\mathbb{Var}(\text{rg}(\hat{y}))}}$ $\in$ $[-1,1]$

```python
from sklearn.metrics import (
    mean_squared_error, # MSE
    mean_absolute_error, # MAE
    mean_absolute_percentage_error, # MAPE
    r2_score # R^2
)

from scipy.stats import spearmanr
import numpy as np

# y_test = true target values
# y_pred = model predictions

# 1. SSE = Sum of Squared Errors
# p_SSE = sum (y_i - prediction_i)^2
SSE = np.sum((y_test - y_pred)**2)


# 2. MSE = Mean Squared Error
# p_MSE = (1/m) * sum (y_i - prediction_i)^2
MSE = mean_squared_error(y_test, y_pred)

# 3. RMSE = Root Mean Squared Error
# p_RMSE = sqrt(MSE)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))

# 4. MAE = Mean Absolute Error
# p_MAE = (1/m) * sum |y_i - prediction_i|
MAE = mean_absolute_error(y_test, y_pred)


# 5. MAPE = Mean Absolute Percentage Error
# ρ_MAPE = (1/m) * sum |(y_i - prediction_i)/y_i|
MAPE = mean_absolute_percentage_error(y_test, y_pred)


# 6. R^2 = Coefficient of Determination
# p_R² = 1 - SSE/SST
R2 = r2_score(y_test, y_pred)

# 7. Spearman Rank Correlation
# ρ_Spearman = corr(rank(y), rank(prediction_i))
Spearman_corr, p_value = spearmanr(y_test, y_pred)
```


<a class="anchor" id="classification"></a>
## 6.2 Classification

* Accuray: $\rho_\text{ACC}=\frac{1}{m} \sum_{i=1}^m \left[y^{(i)} = \hat{y}^{(i)}\right] \in [0,1]$

* Missclassification error: $\rho_\text{MCE}=1-\rho_\text{ACC}=\frac{1}{m} \sum_{i=1}^m \left[y^{(i)} \neq \hat{y}^{(i)}\right] \in [0,1]$

* Brier score: $\rho_\text{BS}=\frac{1}{m}\sum_{i=1}^m (\hat{\pi}^{(i)}-y^{(i)})^2$

* Log loss: $\rho_\text{LL}=\frac{1}{m}\sum_{i=1}^m -y^{(i)} \log(\hat{\pi}^{(i)}) - (1-y^{(i)}) \log(1-\hat{\pi}^{(i)})$

* Confusion matrix: 
$$
\begin{array}{ccc}
&& & y &  \\
&& \color{green}+ &  & \color{red}-  \\
\hline 
&\color{green}+& \text{True positive (TP)} &  & \text{False positive (FP)}  \\
\hat{y}&&&& \\
&\color{red}-& \text{False negative (FN)} &  & \text{True negative (TN)}  \\
\end{array}
$$

$$
\text{Precision}=\frac{\text{TP}}{\text{TP}+\text{FP}}, \quad
\text{Recall}=\frac{\text{TP}}{\text{TP}+\text{FN}}
$$

- Costs: $\frac{1}{n} \sum_{i=1}^n C[y^{(i)},\hat{y}^{(i)}]$

- $\rho_{\text{BS,MC}}=\frac{1}{m} \sum_{i=1}^m \sum_{k=1}^g (\hat{\pi}_k^{(i)}-o_k^{(i)})^2$, $\underbrace{o_k^{(i)}=[y^{(i)}=k]}_{\text{one-hot-encoding}}$

* ROC (Receiver operating characteristics) metrics: 
$$
\begin{array}{ccc}
&& & y &  \\
&& \color{green}+ &  & \color{red}- & \\
\hline 
&\color{green}+& \text{TP} &  & \text{FP} & \rho_{\text{PPV}}  \\
\hat{y}&&&& \\
&\color{red}-& \text{FN} &  & \text{TN} & \rho_{\text{NPV}} \\
\hline 
&& \rho_{\text{TPR}} &  & \rho_{\text{TNR}} & \\
\end{array}
$$

$$
\rho_{\text{PPV}}=\frac{\text{TP}}{\text{TP + FP}}, \quad
\rho_{\text{TPR}}=\frac{\text{TP}}{\text{TP + FN}} \\[3mm]
\rho_{\text{NPV}}=\frac{\text{TN}}{\text{FN + TN}}, \quad
\rho_{\text{TNR}}=\frac{\text{TN}}{\text{FP + TN}} \\
$$

```python
from sklearn.metrics import (
    accuracy_score, # Accuracy
    brier_score_loss, # Brier Score
    log_loss, # Log Loss
    confusion_matrix, # TP, FP, FN, TN
    precision_score, # Precision = PPV
    recall_score, # Recall = TPR
    roc_auc_score
)
import numpy as np

# y_test      = true class labels (0/1 or multiclass)
# y_pred      = predicted hard labels
# y_proba     = predicted probabilities for positive class
# y_proba_mc  = predicted probability matrix for multiclass shape (m,g)


# 1. Accuracy
# p_ACC = (1/m) sum [y_i = prediction_i]
ACC = accuracy_score(y_test, y_pred)

# 2. Misclassification Error
# p_MCE = 1 - Accuracy
MCE = 1 - accuracy_score(y_test, y_pred)


# 3. Brier Score (binary probabilistic predictions)
# ρ_BS = (1/m) sum (prob_i - y_i)^2
BS = brier_score_loss(y_test, y_proba)


# 4. Log Loss / Cross Entropy
# p_LL = (1/m) sum -y_i log(prob_i) - (1-y_i)log(1-prob_i)
LL = log_loss(y_test, y_proba)


# 5. Confusion Matrix
# [[TN, FP],
#  [FN, TP]]
TN, FP, FN, TP = confusion_matrix(y_test, y_pred).ravel()


# 6. Precision = Positive Predictive Value (PPV)
# TP / (TP + FP)
Precision = precision_score(y_test, y_pred)


# 7. Recall = True Positive Rate (TPR, Sensitivity)
# TP / (TP + FN)
Recall = recall_score(y_test, y_pred)


# 8. Negative Predictive Value (NPV)
# TN / (TN + FN)
NPV = TN / (TN + FN)


# 9. True Negative Rate (TNR, Specificity)
# TN / (TN + FP)
TNR = TN / (TN + FP)


# 10. Cost-sensitive classification
# (requires user-defined cost matrix C)
# C[a,b] = cost of predicting b when truth is a
C = np.array([
    [0, 5],   # true class 0 predicted as 0/1
    [10, 0]   # true class 1 predicted as 0/1
])

Costs = np.mean([C[y_true, y_hat] for y_true, y_hat in zip(y_test, y_pred)])


# 11. Multiclass Brier Score
# ρ_BS,MC = (1/m) sum sum (prob_ik - o_ik)^2
one_hot = np.eye(y_proba_mc.shape[1])[y_test]

BS_MC = np.mean(np.sum((y_proba_mc - one_hot)**2, axis=1))


# 12. ROC-AUC (threshold-independent ranking metric)
ROC_AUC = roc_auc_score(y_test, y_proba)
```


#### Example Costs

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/9.png" width="500"/>
  <img src="pics/10.png" width="500"/>
</div>


<a class="anchor" id="resampling"></a>
# 7. Resampling

- $\widehat{\text{GE}}(\hat{f},L):=\frac{1}{m} \sum_{(x,y) \in \mathcal{D}_\text{test}} \left[L\left(y,\hat{f}(x)\right)\right] $

- $\widehat{\text{GE}}(\hat{f},L)$ unbiased on $\mathcal{D}_\text{test}$ and for i.i.d. $\mathcal{D}_\text{test}$

**Problem:**
- If test set $\mathcal{D}_\text{test}$ is small then $\widehat{\text{GE}}(\hat{f},L):= \frac{1}{m} \sum_{(x,y) \in \mathcal{D}_\text{test}} \left[L\left(y,\hat{f}(x)\right)\right]$ will have high variance (very different values for slightly different small test sets $\mathcal{D}_\text{test}$)

**Possible solutions to decrease variance:**
- Increase test set size $m = |\mathcal{D}_\text{test}|$ $\Rightarrow$ means decreasing size of $\mathcal{D}_\text{train}$ which is bad &#10060;
- Compute $\widehat{\text{GE}}(\hat{f},L)$ for $B$ test sets $\mathcal{D}_\text{test}$ and aggregate them &#9989;
    - B test sets: $\mathcal{J}=\underbrace{((\mathcal{J}_\text{train1},\underbrace{\mathcal{J}_\text{test1}}_{\widehat{\text{GE}}(\hat{f},L)_1}),(\mathcal{J}_\text{train2},\underbrace{\mathcal{J}_\text{test2}}_{\widehat{\text{GE}}(\hat{f},L)_2}),\ldots,(\mathcal{J}_\text{trainB},\underbrace{\mathcal{J}_\text{testB}}_{\widehat{\text{GE}}(\hat{f},L)_B}))}_{\text{Aggregate the estimates}}$

- Procedure: Repeatedly split dataset in training and testset $\Rightarrow$ Calculate test errors $\Rightarrow$ Average over test errors
    - Reduced variance from small test sets via averaging over multiple test errors


## Strategies

Goal: Generate $\mathcal{J}$ by splitting full dataset repeatedly into train and test set and train model on the respective train set and evaluate on test set.

<a class="anchor" id="hold_out"></a>
### 7.1 Hold-out sampling

<div style="
<-- -->background-color:rgb(239, 201, 50);;
padding:16px;
border-radius:8px;
color:white;
">

#### Hold-out

- Split dataset once into train and test set (e.g. 70 to 30)
- Train model on train set and evaluate $\widehat{\text{GE}}$ on test set
- Problem: $\widehat{\text{GE}}$ depends heavily on this one random split

<div style="
<-- -->background-color:rgb(213, 230, 222);
padding:14px;
border-radius:8px;
color:white;
margin-top:12px;
margin-left:24px;
width:85%;
border-left:4px solid rgb(208, 215, 212);
box-shadow: inset 0 0 8px rgba(219, 207, 207, 0.3);
">

#### Repeated Hold-out / Subsampling
- Repeat hold-out process multiple times with different random splits
- Each repetition:
    - 1. Randomly split into train/test set (often same ratio each time)
    - 2. Train model on train set
    - 3. Compute test error on test set
- After many repetitions average test errors $\rightarrow$ Stable $\text{GE}$ estimate


</div>
</div>

<p align="center">
<img src="pics\4.png" width="600"/>
</p>

<a class="anchor" id="CV"></a>
### 7.2 k-fold Cross Validation (CV)

- Split data $\mathcal{D}$ into $k$ roughly equally-sized partitions
- Each partition of $\mathcal{D}$ is test set $\mathcal{D}_\text{test}$ once
- Data not in $\mathcal{D}_\text{test}$ is used for training as $\mathcal{D}_\text{train}$
- Aquire $k$ test errors on each test set $\mathcal{D}_\text{test}$ and average them
- Of the data $\mathcal{D}$ approximately $\frac{k-1}{k}$% is used for training

<p align="center">
<img src="pics\5.jpg" width="600"/>
</p>

- More Folds $\Rightarrow$ $\widehat{\text{GE}}$ more unbiased
- k=5 or k=10 folds are common

<a class="anchor" id="CV_start"></a>
### 7.3 Cross-Validation-Stratification

- Used when target classes are very imbalanced
- Preserve distribution of target (or any feature) in each fold

<p align="center">
<img src="pics\6.png" width="600"/>
</p>

**Example:**
- Assume binary classification:
- class ``+`` = 9 observations $\Rightarrow$ 75% positive
- class ``-`` = 3 observations $\Rightarrow$ 25% negative
- total 12 observation $\Rightarrow$ 3-fold CV with 4 observations per fold optimal here

If we randomly split into 3 folds, one fold might accidentally get:
- 4 positives, 0 negatives and another
- 2 positives, 2 negatives

This makes folds unbalanced and validation scores then fluctuate because some folds are artificially easier/harder.

Stratified splitting means:
- preserve original class ratio inside each fold
- overall ratio is 75/25
- each fold of size 4 should ideally contain:
    - 3 positives
    - 1 negative
    
| Fold   | Positive | Negative |
| ------ | -------- | -------- |
| Fold 1 | 3        | 1        |
| Fold 2 | 3        | 1        |
| Fold 3 | 3        | 1        |

**Now every validation round sees similar class composition.**

**Round 1:** <br>

$\quad$ train on folds 2+3, test on fold1

**Round 2:** <br>

$\quad$ train on folds 1+3, test on fold2

**Round 3:** <br>

$\quad$ train on folds 1+2, test on fold3

Final CV error: $\text{CV}_3 = \frac{1}{3}(\text{Test error}_1+\text{Test error}_2+\text{Test error}_3)$


<a class="anchor" id="LOO-CV"></a>
### 7.4 Leave-one-out CV (LOO-CV)

- $(x_i,y_i) = \left((x_{i1},x_{i2},\ldots,x_{ip}),y_i\right)$ for $i=1,\ldots,n$

$$
\left.
\begin{array}{rcl}
\text{Iter 1:}& \text{data }\mathcal{D} &\{\overbrace{{\color{green}{(x_1,y_1)}},{\color{green}{(x_2,y_2)}},{\color{green}{(x_3,y_3)}},\ldots,{\color{green}{(x_{n-1},y_{n-1})}}}^{\color{green}\mathcal{D}_\text{train}},\overbrace{{\color{red}{(x_n,y_n)}}}^{\color{red}\mathcal{D}_\text{test}}\} &\Rightarrow \text{Test error 1} \\

\text{Iter 2:}& \text{data }\mathcal{D} &\{{\color{red}{(x_1,y_1)}},{\color{green}{(x_2,y_2)}},{\color{green}{(x_3,y_3)}},\ldots,{\color{green}{(x_{n-1},y_{n-1})}},{\color{green}{(x_n,y_n)}}\} &\Rightarrow \text{Test error 2} \\ 

\text{Iter 3:}& \text{data }\mathcal{D} &\{{\color{green}{(x_1,y_1)}},{\color{red}{(x_2,y_2)}},{\color{green}{(x_3,y_3)}},\ldots,{\color{green}{(x_{n-1},y_{n-1})}},{\color{green}{(x_n,y_n)}}\} &\Rightarrow \text{Test error 3} \\

\vdots & & \vdots & \vdots \\

\text{Iter n:}& \text{data }\mathcal{D} &\{{\color{green}{(x_1,y_1)}},{\color{green}{(x_2,y_2)}},{\color{green}{(x_3,y_3)}},\ldots,{\color{red}{(x_{n-1},y_{n-1})}},{\color{green}{(x_n,y_n)}}\} &\Rightarrow \text{Test error n} \\

\end{array}
\right\} \text{Average}
$$

**LOO $\Rightarrow$ High variance**

<a class="anchor" id="LOOO"></a>
### 7.5 Leave-one-object-out

**Situation:** We have clusters of observations from the same objects
- e.g. multiple medical images from same patient

**Problem:** 
- Data is no longer i.i..d.
- Data from same object might appear in both train and testset $\Rightarrow$ Test error unrealistically low (bias $\widehat{\text{GE}}$)
- Data from same object should either be in train or testset

<p align="center">
<img src="pics\7.png" width="600"/>
</p>

<a class="anchor" id="bootstrap"></a>
### 7.6 Bootstrap

- $\mathcal{D}$=data set of size $n$
- Draw $B$ trainsets $\mathcal{D}_\text{train}^b$ of size $n$ with replacement from $\mathcal{D}$
- Because of replacement, some data points appear multiple times in the same set
- Train model on $\mathcal{D}_\text{train}^b$ and test on Out-of-Bag (OOB) points $\mathcal{D}_\text{test}^b=\mathcal{D}$ \ $\mathcal{D}_\text{train}^b$
- Each bootstrap sample $\mathcal{D}_\text{train}^b$ of size $n$ includes on average $1-P((x,y) \notin \mathcal{D}_\text{train})=1-(1-\frac{1}{n})^n \overset{n \rightarrow \infty}{\longrightarrow}1-\frac{1}{e} \approx 63.2$% unique points
<p align="center">
<img src="pics\8.png" width="400"/>
</p>


<a class="anchor" id="bootstrap"></a>
#### Short Guideline

- 5-fold or 10-fold CV is standard
- For small $n$ do not use hold-out, CV with few folds or Subsampling with small split rate
- For $n < 200$ use Leave-one-out or repeated CV
- Subsampling often better than bootstrapping 

<a class="anchor" id="metric"></a>
# 8. Choosing the right metric

- No single "best" metric
- Depends on application context:
    - Medical $\rightarrow$ prioritize recall (TPR)
    - Finance $\rightarrow$ balance risk vs reward
- Different metrics $\rightarrow$ different model rankings